<div style='background:#1a1a2e;color:#eee;padding:18px;border-radius:8px;margin-bottom:12px'>
<h2 style='margin:0'>🍎 EDUCATOR VERSION — Chapter 11: Understanding Randomness</h2>
<p style='margin:6px 0 0'>This notebook contains answer keys, teaching notes, and discussion prompts. Students should use the standard <strong>Chapter_11.ipynb</strong>.</p>
</div>


# Chapter 11: Understanding Randomness in Structural Engineering
## Why No Two Steel Beams Are Exactly the Same

**Learning Objectives:**
- Explain what makes a process random vs. predictable
- Demonstrate the Law of Large Numbers using structural material tests
- Distinguish short-run variability from long-run stability
- Connect randomness in material properties to how engineers design with safety factors


<center>
<img src='https://upload.wikimedia.org/wikipedia/commons/thumb/8/89/Compressive_strength_testing_machine.jpg/640px-Compressive_strength_testing_machine.jpg' width='480' />

<em>A concrete cylinder being crushed in a compression testing machine (ASTM C39). Structural engineers test batches of cylinders to estimate actual strength — a direct application of the Law of Large Numbers. (Wikimedia Commons — CC BY-SA.)</em>
</center>

---


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Context

**Curriculum connections:**
- *Statistics class*: Law of Large Numbers; short-run variability vs. long-run stability; sampling variability
- *Physics class*: Measurement uncertainty; material properties and tolerances; why engineers use safety factors

**Prerequisites:** Students should understand basic descriptive statistics (mean, standard deviation) and have been introduced to the idea that repeated measurements vary.

**Estimated time:** 45–55 minutes. Widget 1 (steel beam variability) takes about 20 minutes; Widget 2 (concrete cylinder tests / ACI acceptance) takes about 25 minutes.

**Pedagogical note:** The most common student misconception about randomness is the *gambler's fallacy* — the belief that after several low values, the next must be high. Widget 1 is specifically designed to surface this. Run it with n=5, ask students to predict the next result, then bump to n=50 and n=200 to show that the mean stabilizes without any 'correction.' The concrete cylinder Widget 2 then connects the same idea to a real quality-control decision students can reason about: would you accept or reject this batch?
</div>


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
np.random.seed(11)

# ASTM A572 Grade 50 structural steel — nominal yield strength 50 ksi
# NOTE: The values below (mean ~55 ksi, SD ~3.5 ksi) are illustrative, not from Hibbeler directly.
# They reflect realistic ranges reported in structural reliability literature (e.g., AISC LRFD calibration
# studies). Hibbeler references this steel grade but does not tabulate statistical distributions.
TRUE_MEAN_KSI = 55.0
TRUE_SD_KSI   = 3.5
POPULATION    = np.random.normal(TRUE_MEAN_KSI, TRUE_SD_KSI, 10000)
print('ASTM A572 Grade 50 steel population (10,000 simulated mill coupons)')
print('  [Values are illustrative; see AISC LRFD calibration studies for sources]')
print(f'  Nominal minimum yield strength: 50 ksi')
print(f'  Simulated true mean:  {TRUE_MEAN_KSI} ksi')
print(f'  Simulated true SD:    {TRUE_SD_KSI} ksi')
print(f'  Samples below 50 ksi: {(POPULATION < 50).mean()*100:.1f}%  (these would fail the spec)')


## 11.1  Randomness in Structural Materials

When a steel mill produces a W18×97 wide-flange beam, every beam comes out slightly different. The rolling process, cooling rate, and chemical composition of each heat of steel introduce unavoidable variation. This is not sloppy manufacturing — it is physical reality.

**ASTM A572 Grade 50 steel** (one of the most common structural steels, discussed throughout Hibbeler) has a *nominal* minimum yield strength of **50 ksi**. The statistical values used here — mean ~55 ksi, SD ~3.5 ksi — are *illustrative*, based on ranges reported in structural reliability research. Hibbeler references A572 steel throughout but does not tabulate its statistical distribution; that data comes from AISC calibration studies. But real mill data shows that:

- The **actual mean** yield strength is closer to **55 ksi** — mills intentionally produce above the minimum
- Individual coupon tests scatter around that mean with a standard deviation of about **3–4 ksi**
- A small percentage of coupons will test *below* 50 ksi even from a compliant heat

This is why structural codes require testing **multiple specimens**, not just one. A single test result is random. The average of many tests is not.

That principle — that averages stabilize even when individual results do not — is called the **Law of Large Numbers**.


## 🔬 Interactive Experiment 1: The Law of Large Numbers in a Steel Mill

Imagine you are a quality-control engineer. You pull steel coupons from a production run and test their yield strength. Each test gives you a random result.

- With **1 test**, your estimate of the batch's average strength could be wildly off.
- With **50 tests**, your estimate is much more reliable.
- With **200 tests**, you have a very accurate picture.

Run the simulation below and watch how the running average of coupon tests converges toward the true mean as the number of tests increases.


In [ ]:
def _lln_plot(n_tests):
    sample = np.random.choice(POPULATION, size=n_tests, replace=False)
    running_avg = np.cumsum(sample) / np.arange(1, n_tests + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: individual test results
    ax1.scatter(range(1, n_tests+1), sample, color='steelblue', alpha=0.5, s=15, label='Individual test')
    ax1.axhline(TRUE_MEAN_KSI, color='black', lw=2, ls='--', label=f'True mean: {TRUE_MEAN_KSI} ksi')
    ax1.axhline(50, color='firebrick', lw=1.5, ls=':', label='ASTM minimum: 50 ksi')
    ax1.set_xlabel('Test number', fontsize=12)
    ax1.set_ylabel('Yield Strength (ksi)', fontsize=12)
    ax1.set_title(f'Individual Coupon Tests  (n = {n_tests})', fontsize=13)
    ax1.legend(fontsize=10)

    # Right: running average
    ax2.plot(range(1, n_tests+1), running_avg, color='steelblue', lw=2, label='Running average')
    ax2.axhline(TRUE_MEAN_KSI, color='black', lw=2, ls='--', label=f'True mean: {TRUE_MEAN_KSI} ksi')
    ax2.fill_between(range(1, n_tests+1),
        TRUE_MEAN_KSI - TRUE_SD_KSI, TRUE_MEAN_KSI + TRUE_SD_KSI,
        alpha=0.12, color='steelblue', label='±1 SD band')
    ax2.set_xlabel('Number of Tests', fontsize=12)
    ax2.set_ylabel('Running Average Yield Strength (ksi)', fontsize=12)
    ax2.set_title('Running Average Converges to True Mean', fontsize=13)
    ax2.legend(fontsize=10)
    final_err = abs(running_avg[-1] - TRUE_MEAN_KSI)
    ax2.annotate(f'Error after {n_tests} tests: {final_err:.2f} ksi',
        xy=(0.03, 0.08), xycoords='axes fraction', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    plt.tight_layout()
    plt.show()

w_n = widgets.IntSlider(value=20, min=1, max=300, step=1,
    description='Number of tests:', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out1 = widgets.interactive_output(_lln_plot, {'n_tests': w_n})
display(widgets.VBox([w_n, out1]))


---
## ⚠️  Why Testing Standards Require Multiple Specimens

Before ASTM formalized multi-specimen testing requirements, some engineers accepted single-coupon test results as proof of material compliance. This was dangerous precisely because of the Law of Large Numbers — or rather, its inverse:

> *A single test result tells you almost nothing about the true mean of the population.*

**ASTM A6** (the standard governing structural steel) now requires a minimum number of tension tests per heat of steel, and specifies that the reported yield strength must satisfy statistical requirements — not just a single pass.

The **Quebec Bridge collapse of 1907** is a sobering early example. The bridge's chief engineer, Theodore Cooper, approved the use of heavier steel members based on limited material tests and calculations he performed remotely — without physically being at the site. When a bottom chord member buckled and the bridge collapsed (killing 75 workers), the investigation found that the actual dead load of the structure had been systematically underestimated from the start. A random sample of a few members had been tested and deemed acceptable; the full population of members had not been.

> *Randomness does not go away when you ignore it. It simply waits.*


<div style='background:#d4edda;padding:15px;border-left:5px solid #28a745;margin:10px 0'>

### ✅ Answer Key — Stop and Think (Widget 1)

**1. Running mean at n=5 vs. n=200:** At n=5, the running mean swings wildly — often ±5–10 ksi from the true mean (~55 ksi). By n=200, it has settled to within ~1 ksi. This is the Law of Large Numbers in action: with more trials, the sample mean converges toward the population mean.

**2. Why can't engineers just test one beam?** Because one measurement captures only one draw from the distribution. It might be unusually strong (overconfident) or unusually weak (too conservative). Testing many specimens gives a reliable estimate of the *distribution* — both the mean and the spread — which is what safety factors are calibrated against.

**3. What happens to spread as n increases?** The individual yield strength values keep varying (the distribution doesn't change), but the *running mean* becomes more stable. Students sometimes confuse 'the mean converges' with 'the measurements become more consistent' — they do not. Each new beam is still drawn from the same variable population.

**Common misconception to address:** If the first 10 beams all test above 57 ksi, does the next one have to be lower? No. Each beam is independent. There is no 'correction' mechanism. This is the gambler's fallacy.
</div>


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Note — Quebec Bridge Case Study

The Quebec Bridge (1907) is one of the clearest historical examples of *not taking variability seriously* in structural engineering. The design load calculations used the nominal steel weight (400 tons), but as construction progressed and the actual weight grew toward 1800 tons, the chief engineer Theodore Cooper continued to approve work without field verification.

**Key point for students:** This is not just a calculation error — it is a failure of statistical thinking. Cooper treated a growing, uncertain quantity (the actual structure weight) as if it were a fixed known value. The equivalent in statistics: treating a sample mean as if it were the population mean, without accounting for uncertainty.

**Physics connection:** The collapse was a compression failure in the lower chord near the anchor arm. The actual stress (σ = F/A) exceeded the material's yield strength because F (the actual load) was far larger than F (the assumed design load). The formula is simple; the failure was in the *inputs*, not the math.

**Emotional note:** 75 workers died. This is still the deadliest construction accident in Canadian history. Frame the discussion around the professional responsibility created by uncertainty, not just the math.
</div>


## 🔬 Interactive Experiment 2: Short-Run Surprises in Concrete Cylinder Tests

When a structural concrete pour is placed, ASTM C39 requires engineers to cast and crush companion cylinders to verify that the concrete reached its specified compressive strength f'c. Each cylinder gives a random result — and with only a few tests, the sample mean bounces around.

The simulation below shows what happens as more cylinders are tested from the same pour. Watch how the running maximum (the strongest cylinder seen so far) stabilizes, and how the distribution of strengths takes shape.


<div style='background:#d4edda;padding:15px;border-left:5px solid #28a745;margin:10px 0'>

### ✅ Answer Key — Stop and Think (Widget 2 — Concrete Cylinder Tests)

**1. ACI 318 acceptance:** The ACI 318 standard requires that: (a) no individual cylinder test falls below f'c − 500 psi, AND (b) the average of any three consecutive tests meets or exceeds f'c. The widget shows when these criteria are violated. A batch with many failures near the spec limit *should* trigger a rejection review, even if the average looks acceptable.

**2. Why test multiple cylinders per batch?** Same reason as Widget 1: one cylinder could be an outlier. The ACI criterion uses averages of three consecutive tests precisely to reduce the chance of accepting a genuinely weak batch because one test happened to be high.

**3. Connection to safety factors:** The specified strength f'c = 4000 psi is the *minimum* the engineer assumes in design. The production mean (~4600 psi) is deliberately set higher so that even the lower tail of the distribution stays above 4000 psi. This margin is not waste — it is a direct consequence of variability.
</div>


In [ ]:
# Concrete cylinder compression tests — ASTM C39
# Specified f'c = 4000 psi (a common value for structural slabs and beams)
# Typical production: mean ~4600 psi (batching plant overshoots to meet spec),
# SD ~380 psi (coefficient of variation ~8%, typical for ready-mix concrete)
CONC_FC_SPEC = 4000   # psi, specified minimum compressive strength
CONC_MEAN    = 4600   # psi, typical production mean
CONC_SD      = 380    # psi, typical standard deviation
CONC_POP     = np.abs(np.random.normal(CONC_MEAN, CONC_SD, 50000))

def _cylinder_plot(n_tests):
    sample = np.random.choice(CONC_POP, size=n_tests, replace=True)
    running_max = np.maximum.accumulate(sample)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(range(1, n_tests+1), running_max, color='darkorange', lw=1.5)
    ax1.axhline(np.percentile(CONC_POP, 99), color='firebrick', lw=2, ls='--',
        label='99th percentile strength')
    ax1.axhline(CONC_FC_SPEC, color='gray', lw=1.5, ls=':',
        label=f"Specified f'c = {CONC_FC_SPEC} psi")
    ax1.set_xlabel('Number of Cylinder Tests', fontsize=12)
    ax1.set_ylabel("Compressive Strength (psi)", fontsize=12)
    ax1.set_title('Running Maximum — Strongest Cylinder Seen So Far', fontsize=13)
    ax1.legend(fontsize=10)

    ax2.hist(sample, bins=40, color='darkorange', edgecolor='white', alpha=0.75)
    ax2.axvline(np.mean(sample), color='black', lw=2, ls='--',
        label=f'Sample mean: {np.mean(sample):.0f} psi')
    ax2.axvline(CONC_FC_SPEC, color='firebrick', lw=2, ls=':',
        label=f"Specified f'c = {CONC_FC_SPEC} psi")
    pct_below = (sample < CONC_FC_SPEC).mean() * 100
    ax2.set_xlabel("Compressive Strength f'c (psi)", fontsize=12)
    ax2.set_ylabel('Number of Cylinders', fontsize=12)
    ax2.set_title(f'Distribution of {n_tests} Cylinder Tests', fontsize=13)
    ax2.legend(fontsize=10)
    ax2.annotate(f"{pct_below:.1f}% of cylinders\nbelow specified f'c",
        xy=(0.03, 0.80), xycoords='axes fraction', fontsize=10,
        bbox=dict(boxstyle='round',
            facecolor='mistyrose' if pct_below > 10 else 'lightgreen', alpha=0.9))
    plt.tight_layout()
    plt.show()

w_n2 = widgets.IntSlider(value=50, min=5, max=500, step=5,
    description='Cylinder tests:', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out2 = widgets.interactive_output(_cylinder_plot, {'n_tests': w_n2})
display(widgets.VBox([w_n2, out2]))


---
## 📋  Chapter 11 Review

| Term | Meaning |
|------|--------|
| **Random process** | Individual outcomes are unpredictable; the long-run pattern is stable |
| **Law of Large Numbers** | As sample size grows, the sample mean converges to the true population mean |
| **Short-run variability** | Individual results fluctuate widely around the true mean |
| **Long-run stability** | The average of many results settles near the true mean |
| **Simulation** | Using a computer to repeat a random process many times and observe the pattern |

**The Big Idea:** Structural engineers rely on the Law of Large Numbers every time they use a code-specified load or material strength. Those values were derived from thousands of tests and measurements. No single test result tells you much — but the average of many tests is a reliable guide for design. Randomness, understood correctly, is not a source of danger. Ignoring it is.


<div style='background:#d1ecf1;padding:15px;border-left:5px solid #17a2b8;margin:10px 0'>

### 💬 Discussion Prompts

**1. Gambler's fallacy (warm-up):** A fair coin has come up heads 8 times in a row. What is the probability the next flip is tails? *(Answer: 50% — each flip is independent.) How does this connect to running average of beam strengths? Both demonstrate that past results don't 'load up' a correction in a random process.*

**2. Sample size trade-offs (pair discussion):** Testing more concrete cylinders gives a better estimate of batch strength — but tests cost time and money. If you were the structural engineer of record for a new hospital, how many cylinders per batch would you want? How does the stakes of the project affect your answer? *(Guide toward: higher consequence → more testing. This is the cost-benefit logic behind sampling theory.)*

**3. Material consistency (small group):** Modern steel is produced in large continuous casts with tight quality control. Concrete is mixed on-site from local materials that vary by season and supplier. Which material would you expect to have a *smaller* coefficient of variation (CV = σ/μ)? Why? *(Steel typically CV ≈ 5–7%; concrete CV ≈ 8–12% for compressive strength.) What does this imply for safety factors?*

**4. Extension (homework):** The Law of Large Numbers is sometimes misunderstood as meaning 'things even out over time.' Find an example from sports statistics, medicine, or finance where someone made a decision based on the gambler's fallacy. Explain the error and what the LLN actually predicts.
</div>
